In [ ]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [ ]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

In [ ]:
# custom
from utils import *

# LOAD LETTERS

In [ ]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [ ]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [ ]:
word_df['word'] = word_df['word'].astype(str)

In [ ]:
word_df.head()

In [ ]:
word_df.shape

In [ ]:
word_df['word'].isna().value_counts()

In [ ]:
word_df['lcase'] = word_df['word'].str.lower()

In [ ]:
word_df['n_letters'] = word_df['word'].str.len()

In [ ]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))

In [ ]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [ ]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [ ]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [ ]:
word_df.head()

In [ ]:
word_df.shape

In [ ]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

In [ ]:
word_df['word_id'] = range(0, word_df.shape[0])

In [ ]:
word_df.shape

In [ ]:
word_df.head()

In [ ]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

# BYTE ENCODE WORDS

In [ ]:
word_df['word_byte'] = word_df['word'].map(byte_encode_words)

In [ ]:
word_df.head()

In [ ]:
word_byte_list = word_df['word_byte'].tolist()

## EXAMPLES OF BYTE COMPARISONS

In [ ]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [ ]:
# no letters in common
w1b & w2b

In [ ]:
# letters in common
w1b & w3b

In [ ]:
w1b | w2b

In [ ]:
# this is the same as directly above
testo = byte_encode_words('abhorcleft')
testo

In [ ]:
lc_be = byte_encode_words(ascii_lowercase)

In [ ]:
lc_be

In [ ]:
word_byte_array = np.array(word_byte_list, dtype = np.int32)

In [ ]:
word_byte_to_word_dict = {wb:lcase for wb, lcase in zip(word_df['word_byte'], word_df['lcase'])}

In [ ]:
# do it all at once
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

In [ ]:
# so, now, let's try computing all possible pairs
total_output = []
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## build l4
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## build l5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = [w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5]
                    total_output.append(temp_list)


    if i_row % 10000 == 0:
        print(i_row)
        print(total_output.shape)
    


In [ ]:
l5_df = pd.DataFrame(data = total_output, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5'])


In [ ]:
l5_df.head()

# BUILD LEVEL 2 USING COMBINATIONS

In [ ]:
l2_df = build_l2(word_byte_list=word_byte_list)

In [ ]:
l3_df = build_l3(word_byte_array = word_byte_array, l2_df=l2_df)

In [ ]:
l4_df = build_l4(word_byte_array = word_byte_array, l3_df = l3_df)

In [ ]:
l5_df = build_l5(word_byte_array = word_byte_array, l4_df = l4_df)

In [ ]:
# try something else clever...

In [ ]:
l2_df.shape

In [ ]:
l2_array = l2_df['l2'].to_numpy()

# IDENTIFY ALL WORD GROUPS

In [ ]:
l2_df_test = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)
l3_df_test = build_l3(word_byte_array = word_byte_array, l2_df = l2_df, focal_values=True)
l4_df_test = build_l4(word_byte_array = word_byte_array, l3_df = l3_df, focal_values=True)
l5_df_test = build_l5(word_byte_array = word_byte_array, l4_df = l4_df, focal_values=True)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.shape

In [ ]:
for idx in range(1, 3):
    cn = f"w{str(idx)}"
    ncn = f"w{str(idx)}lc"
    l2_df[ncn] = l2_df[cn].map(word_byte_to_word_dict)

In [ ]:
l2_df.head()

In [ ]:
l2_df.to_excel(excel_writer = 'l2_output.xlsx', index = False)

In [ ]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [ ]:
l5_df

In [ ]:
l5_df.to_excel(excel_writer='output_words.xlsx', index = False)

In [ ]:
# START AT LEVEL 2

In [ ]:
l2_set = set(l5_df['l2'].tolist())

In [ ]:
len(l2_set)

In [ ]:
l2_test.shape

In [ ]:
l2_df.shape

In [ ]:
l2_test.shape

In [ ]:
l2_df.to_csv(path_or_buf='l2.txt', sep = '\t', index = False)
l3_df.to_csv(path_or_buf='l3.txt', sep = '\t', index = False)
l4_df.to_csv(path_or_buf='l4.txt', sep = '\t', index = False)
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)

# LOAD IN PREVIOUS OUTPUT

In [ ]:
l2_df = pd.read_csv(filepath_or_buffer='l2.txt', sep = '\t')
l3_df = pd.read_csv(filepath_or_buffer='l3.txt', sep = '\t')
l4_df = pd.read_csv(filepath_or_buffer='l4.txt', sep = '\t')
l5_df = pd.read_csv(filepath_or_buffer='l5.txt', sep = '\t')

In [ ]:
l5_df.shape

In [ ]:
l2_set = set(l5_df['l2'].unique().tolist())

In [ ]:
l2_set

In [ ]:
test_l2_df = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)

In [ ]:
test_l2_df.shape

In [ ]:
# get max valuesa
l2_df.max(axis = 0)

In [ ]:
l2_df['l2'].astype(np.int32).max()

In [ ]:
l3_df.head()

In [ ]:
l3_df.max(axis = 0)

In [ ]:
l4_df.max(axis = 0)

In [ ]:
my_columns = l5_df.columns.tolist()[:9]

In [ ]:
l5_df[my_columns].max(axis = 1)

In [ ]:
# join to get the different word combinations

In [ ]:
l5_df.head()

In [ ]:
l4_df.head()

In [ ]:
l4_df.shape

In [ ]:
l4_df.loc[l4_df['l4'] == 27784191, ]

In [ ]:
output_list = []
for ir5, row5 in l5_df.iterrows():
    l5 = row['l5']
    l5